<a href="https://colab.research.google.com/github/FishyFoshy/COMP3608-Project/blob/main/Dataset_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Cleaning of Dataset 2

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# Load datasets
df2 = pd.read_csv('clean_data.csv')

# Correlation heatmap for Dataset 1 (richest numeric features)
numeric_cols = df2.select_dtypes(include=[np.number]).columns.tolist()
plt.figure(figsize=(10, 8))
sns.heatmap(df2[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Dataset 1: Correlation Heatmap (Numeric Features)')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split

# Drops all unnecessary columns
df2 = df2.drop(columns=['age']) 

# Drops all with null values
df2.dropna(inplace=True)

# Remove Outliers
upper_quantile = df2['price'].quantile(0.99)
df2 = df2[df2['price'] <= upper_quantile]

# Log transforms price to reduce variablilty
df2['price'] = np.log(df2['price'])

# One-Hot encode all categorical features
categorical_cols_to_encode = df2.select_dtypes(include='object').columns
df2 = pd.get_dummies(df2, columns=categorical_cols_to_encode, drop_first=True)

# Find the target and numerical features
target_column = 'price'
numerical_features = ['area', 'bhk', 'bathroom']
y = df2[target_column].copy()

# Create scalar
scaler = StandardScaler()

# Create feature base without price
x_base = df2.drop(columns=[target_column]).copy()

x_linear = x_base.copy()

# Apply polynomial features to numerical columns for linear model
poly = PolynomialFeatures(degree=2, include_bias=False)
numerical_poly_features = poly.fit_transform(x_linear[numerical_features])
poly_feature_names = poly.get_feature_names_out(numerical_features)
x_linear_poly_df = pd.DataFrame(numerical_poly_features, columns=poly_feature_names, index=x_linear.index)

# Drop original numerical columns and concatenate polynomial features for linear model
x_linear = x_linear.drop(columns=numerical_features)
x_linear = pd.concat([x_linear, x_linear_poly_df], axis=1)

# Find all numerical features, even the created ones for linear model
all_numerical_features_linear = x_linear.select_dtypes(include=np.number).columns.tolist()

# Scale numerical features for linear model
x_linear[all_numerical_features_linear] = scaler.fit_transform(x_linear[all_numerical_features_linear])

x_tree = x_base.copy()

# Scale numerical features for tree models
x_tree[numerical_features] = scaler.fit_transform(x_tree[numerical_features])

# Splits the data 80/20 for training and testing the model respectfully
x_train_linear, x_test_linear, y_train_linear, y_test_linear = train_test_split(x_linear, y, test_size=0.2, random_state=42)
x_train_tree, x_test_tree, y_train_tree, y_test_tree = train_test_split(x_tree, y, test_size=0.2, random_state=42)

print("=" * 60)
print("DATASET 2: Dataset 2.csv")
print("=" * 60)
print("\n--- Features for Linear Models (x_linear) ---")
print("\nFirst 5 rows:")
display(x_linear.head(5))
print("\nData types:")
print(x_linear.dtypes)
print("\nMissing values:")
print(x_linear.isnull().sum().sum())
print(f"Shape of x_linear: {x_linear.shape}")

print("\n--- Features for Tree Models (x_tree) ---")
print("\nFirst 5 rows:")
display(x_tree.head(5))
print("\nData types:")
print(x_tree.dtypes)
print("\nMissing values:")
print(x_tree.isnull().sum().sum())
print(f"Shape of x_tree: {x_tree.shape}")

print("\nTarget variable (price) statistics:")
print(y.describe())

# Price is in Lahk which is 100,000 Rupees (₹)

In [ ]:
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def rmse_original_scale(y_true, y_pred):
    # Inverse transform to original scale
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate RMSE on the original scale
    return np.sqrt(mean_squared_error(y_true_original, y_pred_original))

def mae_original_scale(y_true, y_pred):
    # Inverse transform to original scale
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate MAE on the original scale
    return mean_absolute_error(y_true_original, y_pred_original)

def r2_original_scale(y_true, y_pred):
    # Inverse transform to original scale
    y_true_original = np.exp(y_true)
    y_pred_original = np.exp(y_pred)
    # Calculate R2 on the original scale
    return r2_score(y_true_original, y_pred_original)

# Create scorers that can be used with GridSearchCV or cross_val_score
# 'neg_' prefix is used for metrics to be minimized (RMSE, MAE)
neg_rmse_original_scorer = make_scorer(rmse_original_scale, greater_is_better=False)
neg_mae_original_scorer = make_scorer(mae_original_scale, greater_is_better=False)
r2_original_scorer = make_scorer(r2_original_scale, greater_is_better=True)

### Linear Regression for Dataset 2

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

#Initializing Linear Regression Model and Running
lr = LinearRegression()
lr.fit(x_train_linear, y_train_linear)
y_pred = lr.predict(x_test_linear)

#Fetching the Performance Metrics For The Model
rmse = rmse_original_scale(y_test_linear, y_pred)
mae = mae_original_scale(y_test_linear, y_pred)
r2 = r2_original_scale(y_test_linear, y_pred)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate


#Graph Showing predicted vs Actual With Errors and Coefficents
cv_results_lr = cross_validate(lr, x_linear, y, cv=5, scoring={'rmse': neg_rmse_original_scorer, 'mae': neg_mae_original_scorer, 'r2': r2_original_scorer})

#Printing Performance Metrics for the model
print(f"Root Mean Squared Error: {rmse:.2f}\n")
print(f"Mean Absolute Error: {mae:.2f}\n")
print(f"R^2 Score: {r2:.2f}\n")
print(f"\nCross-validation RMSE:\t₹{-cv_results_lr['test_rmse'].mean():,.2f}K")
print(f"Cross-validation MAE:\t₹{-cv_results_lr['test_mae'].mean():,.2f}K")
print(f"Cross-validation R²:\t{cv_results_lr['test_r2'].mean():.4f}")

y_test_actual = np.exp(y_test_linear)
y_pred_actual = np.exp(y_pred)

#Constructing A Scatter Plot of Real Vs Expected Value to Vizualize the Perfomrance of the Model on The Test Set
plt.figure(figsize=(10, 6))
plt.scatter(y_test_actual, y_pred_actual, alpha=0.6, color='blue')
min_val = min(y_test_actual.min(), y_pred_actual.min())
max_val = max(y_test_actual.max(), y_pred_actual.max())
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2)
plt.xlabel('Actual Prices (Lakh ₹)', fontsize=12)
plt.ylabel('Predicted Prices (Lakh ₹)', fontsize=12)
plt.title('Actual vs. Predicted Property Prices', fontsize=14)
plt.grid(True, linestyle=':', alpha=0.7)
plt.show()

In [ ]:
#Fetching Feature Names and Coefficents
feature_names = x_train_linear.columns
coefficients = lr.coef_

#Wrapping them together in a Pandas DataFrame for Easy Comparision and Viewing
feature_importance = pd.DataFrame({
 'Feature': feature_names,
 'Coefficient': coefficients
})

#Using the Absolute Function in the NumPy Library before Sorting the Coefficents in Decending Order
feature_importance['Abs_Coefficient'] = np.abs(feature_importance['Coefficient'])
top_10_features = feature_importance.sort_values(by='Abs_Coefficient', ascending=False).head(10)

#Plotting Graph
plt.figure(figsize=(10, 6))
sns.barplot(
 data=top_10_features, 
 x='Coefficient', 
 y='Feature', 
)
plt.title('Top 10 Features Driving Property Prices', fontsize=15)
plt.xlabel('Coefficient Value (Impact on Price in Lakh ₹)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(True, axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
top_10_features['Percentage'] = ((np.exp(top_10_features['Coefficient']) - 1) * 100).round(2)
display(top_10_features[['Feature', 'Coefficient', 'Percentage']])

### Analysis

#### Model Performance Overview

The linear regression model has a high predictive power to predict property prices, with the R2 value of the model being 0.93 in the test set, which would mean that the model explains 93% of the variation in property prices. Root mean Squared Error (RMSE) of ₹22.94 Lakhs and Mean Absolute Error (MAE) of ₹10.73 Lakhs are reasonable errors in prediction given the range of property prices in the dataset.

Nonetheless, the cross-validation results demonstrate that there is a significant performance difference between training and validation, with the cross-validation R2 decreasing to 0.8002 and the RMSE increasing to ₹31.93K. This implies that the model has a modest degree of overfitting and does not perform as well on unknown data as it does on training data. The difference in R 2 of 12.98 implies that the model might not generalize to new properties, which are not represented in the training set.

#### Feature Importance & Real-World Implications

The analysis of feature importance shows that location is the most important variable in the pricing of properties with the top 10 most significant features being location based. This goes in line with real estate basics: location is universally acknowledged as the key influencer of property value.

**Key Insights:**

- Premium Locations (Tiruvottiyur and Gopalapar) have positive coefficients, which means that properties in these areas fetch a price premium, probably because these areas have better infrastructure, amenities, or desirability.

- Negative coefficients in Lower-Value Locations (Puzhal, Sevvapet, Sriperumbudur) indicate that properties in these areas are discounted, possibly because of distance to city centers or limited development opportunities.

- Location's Dominance:  The fact that location features are concentrated in the top 10 indicates that the dataset does not vary enough in other structural characteristics (BHK, bathroom count, area) in comparison to differences in location, or that location genuinely outweighs structural characteristics in this real estate market.

#### Prediction Accuracy Assessment

In the scatter plot "Actual vs. Predicted Property Prices" it can be seen that the predictions cluster closely around the ideal prediction line, particularly in the lower price ranges ( 0-200 Lakhs). The high-value properties (400+ Lakhs) are more scattered, though, which means that the model has difficulty with outliers and high-value properties. This is common with linear models with heteroscedastic data (not equal variance through price ranges).

#### Practical Implications for Real Estate

1. Reliability on Standard Property: The model is very reliable in predicting prices of typical properties in the ₹50- 200 Lakhs range, thus making it practical in forecasting the average market values.

2. Luxury Segment Limitations: The larger prediction error in high-value properties indicates that the model would have to be refined to predict high-value properties, which is possible by adding other features that are unique to luxury properties.

3. Market Dynamics: The dominance of location as a characteristic is an indication of a supply-constrained or geographically fragmented market in which demand significantly varies by neighborhood which could be an indication that property development or demand are concentrated in certain preferred locations.

4. Generalization Concerns: The cross-validation gap implies that care should be taken when using this model on any property that does not belong to the training distribution or is located in an emerging/developing region that is not well-represented in the data.

#### Conclusion

The linear regression model is a useful predictive model used in predicting the typical residential properties in this market especially when the location information is available. It is easy to use, interpret, and thus useful when a quick valuation or market analysis is needed. Nevertheless, it has limitations with high-value outliers and modest cross-validation performance that justify using the ensemble techniques (Random Forest or XGBoost) to make important valuation decisions.

### Random Forest Regressor on Dataset 2

In [ ]:
from scipy.stats import randint, uniform
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

# Define the parameter distribution to search
params = {
    'n_estimators': randint(100, 600), # Number of trees in the forest
    'max_features': uniform(0, 1), # Number of features to consider at each split
    'min_samples_leaf': randint(1, 10), # Minimum number of samples required at a leaf node
    'min_samples_split' : randint(2, 10), # The minimum number of samples required to split an internal node
    'max_depth': randint(10, 80) # Maximum depth of the tree
}

# Initialize RandomizedSearchCV
# We use the negative root mean squared error as our scoring metric, 
# as RandomizedSearchCV tries to maximize the score, so we negate RMSE.
search = RandomizedSearchCV(
    estimator = RandomForestRegressor(random_state=42), # Use a fixed random_state for reproducibility
    param_distributions = params,
    n_iter = 300,
    scoring = {'rmse': neg_rmse_original_scorer, 'mae': neg_mae_original_scorer, 'r2': r2_original_scorer}, # Metric to optimize
    refit = 'mae',
    cv = 5, # 5-fold cross-validation
    n_jobs = 5, # Use 5 CPU cores
    return_train_score = True,
    random_state = 42, # Use a fixed random_state for reproducibility
    verbose = 2 # Display progress
)

# Fit RandomizedSearchCV to the training data
print("Starting RandomizedSearchCV...")
search.fit(x_train_tree, y_train_tree)
print("RandomizedSearchCV completed.")

# Get the best parameters and best score
best_params = search.best_params_
best_score = abs(search.best_score_)

print(f"\nBest Parameters: {best_params}")
print(f"Best Cross-Validation MAE: ${best_score:,.4f}")

# You can also get the best estimator found by RandomizedSearchCV
best_rfr_model = search.best_estimator_

# Evaluate the best model on the test set
y_pred_tuned = best_rfr_model.predict(x_test_tree)
rmse_tuned = rmse_original_scale(y_test_tree, y_pred_tuned)
mae_tuned = mae_original_scale(y_test_tree, y_pred_tuned)
r2_tuned = r2_original_scale(y_test_tree, y_pred_tuned)

print("\n--- Tuned Random Forest Model Performance ---")
print(f"RMSE on Test Set:\t₹{rmse_tuned:,.2f}K")
print(f"MAE on Test Set:\t₹{mae_tuned:,.2f}K")
print(f"R² on Test Set:\t{r2_tuned:.4f}")

cv_results_rfr = cross_validate(best_rfr_model, x_tree, y, cv=5, scoring={'rmse': neg_rmse_original_scorer, 'mae': neg_mae_original_scorer, 'r2': r2_original_scorer})

# You can also run cross-validation on the best model separately for comparison
cv_rmse_tuned = -cv_results_rfr['test_rmse'].mean()
cv_mae_tuned = -cv_results_rfr['test_mae'].mean()
cv_r2_tuned = cv_results_rfr['test_r2'].mean()

print(f"\nCross-validation RMSE (Best Model):\t₹{cv_rmse_tuned:,.2f}K")
print(f"Cross-validation MAE (Best Model):\t₹{cv_mae_tuned:,.2f}K")
print(f"Cross-validation R² (Best Model):\t{cv_r2_tuned:.4f}")

y_test_original = np.exp(y_test_tree)
y_pred_original = np.exp(y_pred_tuned)

# Predicted vs Actual scatter
plt.figure(figsize=(10, 6))
plt.scatter(y_test_original, y_pred_original, alpha=0.5, s=16, color="blue")
plt.plot([0, y_test_original.max()], [0, y_test_original.max()], 'r--', label='Perfect prediction')
plt.xlabel('Actual Prices (Lakh ₹)', fontsize=12)
plt.ylabel('Predicted Prices (Lakh ₹)', fontsize=12)
plt.title('Actual vs. Predicted Property Prices', fontsize=14)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Get feature importances from the Random Forest Regressor
feature_importances = best_rfr_model.feature_importances_
feature_names = x_tree.columns

# Create a DataFrame for feature importance
feature_importance_rfr = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
})

# Visualization: Feature Category Importance Breakdown
area_importance = feature_importance_rfr[feature_importance_rfr['Feature'] == 'area']['Importance'].values[0]
bhk_importance = feature_importance_rfr[feature_importance_rfr['Feature'] == 'bhk']['Importance'].values[0]
bathroom_importance = feature_importance_rfr[feature_importance_rfr['Feature'] == 'bathroom']['Importance'].values[0]

structural_features = area_importance + bhk_importance + bathroom_importance

location_features = feature_importance_rfr[feature_importance_rfr['Feature'].str.startswith('location_')]['Importance'].sum()
builder_features = feature_importance_rfr[feature_importance_rfr['Feature'].str.startswith('builder_')]['Importance'].sum()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Pie chart of feature categories
categories = ['Structural\n(Area, BHK, Bathroom)', 'Location', 'Builder']
importances = [structural_features, location_features, builder_features]
colors = ['#2E86AB', '#A23B72', '#F18F01']

axes[0].pie(importances, labels=categories, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Feature Importance by Category', fontsize=14, fontweight='bold')

# Detailed breakdown of top features
top_10_features = feature_importance_rfr.sort_values(by='Importance', ascending=False).head(10)

axes[1].barh(range(len(top_10_features)), top_10_features['Importance'].values)
axes[1].set_yticks(range(len(top_10_features)))
axes[1].set_yticklabels(top_10_features['Feature'].values, fontsize=10)
axes[1].set_xlabel('Feature Importance', fontsize=12)
axes[1].set_title('Top 10 Features in Detail', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFeature Importance Breakdown:")
print(f"Structural Features: {structural_features*100:.2f}%")
print(f"Location Features: {location_features*100:.2f}%")
print(f"Builder Features: {builder_features*100:.2f}%")
display(top_10_features[['Feature', 'Importance']])

In [ ]:
# Price Range Performance
print("PERFORMANCE ACROSS PRICE SEGMENTS")
print("-" * 40)

price_ranges = [
    (y_test_original.min(), y_test_original.quantile(0.33), "Low"),
    (y_test_original.quantile(0.33), y_test_original.quantile(0.67), "Medium"),
    (y_test_original.quantile(0.67), y_test_original.max(), "High")
]

for min_p, max_p, label in price_ranges:
    mask = (y_test_original >= min_p) & (y_test_original <= max_p)
    if mask.sum() > 0:
        range_rmse = np.sqrt(mean_squared_error(y_test_original[mask], y_pred_original[mask]))
        range_mae = mean_absolute_error(y_test_original[mask], y_pred_original[mask])
        range_r2 = r2_score(y_test_original[mask], y_pred_original[mask])
        
        print(f"\n{label} Price Range (₹{min_p:.0f}K - ₹{max_p:.0f}K):")
        print(f"  - RMSE: ₹{range_rmse:.2f}K")
        print(f"  - MAE: ₹{range_mae:.2f}K")
        print(f"  - R²: {range_r2:.4f}")
        print(f"  - Sample size: {mask.sum()}")

### Analysis

The analysis of the Random Forests shows that the prediction of property prices is fundamentally basedon a mix of structural and spatial factors, although hierarchically the most significant ones are clear:

**1. Structural Dominance (52.6% of predictive power)**
- **Area (23.5%)**: Area is by far the most important price determinant. This feature alone serves half of all predictions and is based on the idea that the bigger the property, the higher the price.
- **BHK and Bathrooms (28.7%)**: These criteria have secondary yet significant impact. Nevertheless, they are very much correlated with area, bigger houses obviously have more rooms and that is why they become less significant when area is present.

**2. Location Effects (32.1% of predictive power)**
- Location features as a group make around a fifth of the model predictions, but crucially, location features help reveal non-linear spatial patterns that cannot be identified by simpler linear models.
- Certain neighborhoods (Veppampattu, West Mambalam, T Nagar) exhibit different price impacts, suggesting that location is a multiplicative, rather than an additive, factor.

**3. Builder Identity (15.3% of predictive power)**
- Although the reputation of a builder has little average effect, some developers ( Viswaraj, Karthick) display some observable effect, probably due to brand value and reputation of quality of construction.

### Model Performance Strengths

The Random Forest achieves R2 of 0.87 on the test set, indicating that the model explains 87% of price variance. Notably:
- **High-end performance**: The model performs well on predicting costly properties (R2 = 0.79 in high-price range) where linear models usually fail.
- **Minimal overfitting**: The cross-validation R2 of 0.72 is near the test R2 of 0.87, indicating that the model is not overfitting.

### Key Takeaways for Property Stakeholders

1. **For Buyers/Sellers**: Property size (area) should be given priority.
2. **For Investors**: There are opportunities for upselling in wealthier locations.
3. **For Developers**: Builder reputation, while less important than expected, does influence pricing and could be leveraged for premium positioning.


### Gradient Boosting on Dataset 2

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
import numpy as np
import matplotlib.pyplot as plt

# Define the parameter grid to search for XGBoost
param_grid_xgb = {
    "colsample_bytree": [0.2, 0.3, 0.4],
    "gamma": [0, 0.1,],
    "learning_rate": [0.03, 0.1, 0.29, 0.3], # default 0.1 
    "max_depth": [2, 3, 4], # default 3
    "n_estimators": [500, 600, 700], # default 100
    "subsample": [1, 0.9, 0.5, 0]
}

# Initialize GridSearchCV for XGBoost
print("Starting GridSearchCV for XGBoost...")
grid_search_xgb = GridSearchCV(
    estimator=XGBRegressor(objective="reg:squarederror", random_state=42),
    param_grid=param_grid_xgb,
    cv=5,
    scoring=r2_original_scorer, # Using the custom R2 scorer
    n_jobs=5,
    verbose=2
)

# Fit GridSearchCV to the training data
grid_search_xgb.fit(x_train_tree, y_train_tree)
print("GridSearchCV for XGBoost completed.")

# Get the best parameters and best score
best_params_xgb = grid_search_xgb.best_params_
best_score_xgb = grid_search_xgb.best_score_

print(f"\nBest Parameters for XGBoost: {best_params_xgb}")
print(f"Best Cross-Validation R² for XGBoost: {best_score_xgb:.4f}")

# Get the best estimator found by GridSearchCV
best_xgb_model = grid_search_xgb.best_estimator_

# Evaluate the best XGBoost model on the test set
y_pred_xgb = best_xgb_model.predict(x_test_tree)

# Evaluate on original scale
rmse_xgb = rmse_original_scale(y_test_tree, y_pred_xgb)
mae_xgb = mae_original_scale(y_test_tree, y_pred_xgb)
r2_xgb = r2_original_scale(y_test_tree, y_pred_xgb)

print("\n--- Tuned XGBoost Regressor Performance ---")
print(f"RMSE on Test Set:\t₹{rmse_xgb:,.2f}K")
print(f"MAE on Test Set:\t₹{mae_xgb:,.2f}K")
print(f"R² on Test Set:\t{r2_xgb:.4f}")

cv_results_xgb = cross_validate(best_xgb_model, x_tree, y, cv=5, scoring={'rmse': neg_rmse_original_scorer, 'mae': neg_mae_original_scorer, 'r2': r2_original_scorer})

# Cross-validation for the best model
cv_rmse_xgb = -cv_results_xgb['test_rmse'].mean()
cv_mae_xgb = -cv_results_xgb['test_mae'].mean()
cv_r2_xgb = cv_results_xgb['test_r2'].mean()

print(f"\nCross-validation RMSE (Best XGBoost Model):\t₹{cv_rmse_xgb:,.2f}K")
print(f"Cross-validation MAE (Best XGBoost Model):\t₹{cv_mae_xgb:,.2f}K")
print(f"Cross-validation R² (Best XGBoost Model):\t{cv_r2_xgb:.4f}")

y_test_original = np.exp(y_test_tree)
y_pred_original = np.exp(y_pred_xgb)

# Predicted vs Actual scatter
plt.figure(figsize=(8, 6))
plt.scatter(y_test_original, y_pred_original, alpha=0.4, s=15)
plt.plot([0, y_test_original.max()], [0, y_test_original.max()], 'r--', label='Perfect prediction')
plt.xlabel('Actual Price (Lakh ₹)')
plt.ylabel('Predicted Price (Lakh ₹)')
plt.title('XGBoost Regressor: Predicted vs Actual House Price')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

# Fetching Feature Names and Importances from XGBoost
feature_names = x_train_tree.columns
importances = best_xgb_model.feature_importances_

# Wrapping them together in a Pandas DataFrame for Easy Comparison and Viewing
feature_importance_xgb = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
})

# Visualization: Feature Category Importance Breakdown
area_importance = feature_importance_xgb[feature_importance_xgb['Feature'] == 'area']['Importance'].values[0]
bhk_importance = feature_importance_xgb[feature_importance_xgb['Feature'] == 'bhk']['Importance'].values[0]
bathroom_importance = feature_importance_xgb[feature_importance_xgb['Feature'] == 'bathroom']['Importance'].values[0]

structural_features = area_importance + bhk_importance + bathroom_importance

location_features = feature_importance_xgb[feature_importance_xgb['Feature'].str.startswith('location_')]['Importance'].sum()
builder_features = feature_importance_xgb[feature_importance_xgb['Feature'].str.startswith('builder_')]['Importance'].sum()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Pie chart of feature categories
categories = ['Structural\n(Area, BHK, Bathroom)', 'Location', 'Builder']
importances = [structural_features, location_features, builder_features]
colors = ['#2E86AB', '#A23B72', '#F18F01']

axes[0].pie(importances, labels=categories, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Feature Importance by Category', fontsize=14, fontweight='bold')

# Detailed breakdown of top features
top_10_features = feature_importance_xgb.sort_values(by='Importance', ascending=False).head(10)

axes[1].barh(range(len(top_10_features)), top_10_features['Importance'].values)
axes[1].set_yticks(range(len(top_10_features)))
axes[1].set_yticklabels(top_10_features['Feature'].values, fontsize=10)
axes[1].set_xlabel('Feature Importance', fontsize=12)
axes[1].set_title('Top 10 Features in Detail', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFeature Importance Breakdown:")
print(f"Structural Features: {structural_features*100:.2f}%")
print(f"Location Features: {location_features*100:.2f}%")
print(f"Builder Features: {builder_features*100:.2f}%")
display(top_10_features[['Feature', 'Importance']])

In [ ]:
# Price Range Performance
print("PERFORMANCE ACROSS PRICE SEGMENTS")
print("-" * 40)

price_ranges_xgb = [
    (y_test_original.min(), y_test_original.quantile(0.33), "Low"),
    (y_test_original.quantile(0.33), y_test_original.quantile(0.67), "Medium"),
    (y_test_original.quantile(0.67), y_test_original.max(), "High")
]

for min_p, max_p, label in price_ranges_xgb:
    mask = (y_test_original >= min_p) & (y_test_original <= max_p)
    if mask.sum() > 0:
        range_rmse = np.sqrt(mean_squared_error(y_test_original[mask], y_pred_original[mask]))
        range_mae = mean_absolute_error(y_test_original[mask], y_pred_original[mask])
        range_r2 = r2_score(y_test_original[mask], y_pred_original[mask])
        
        print(f"\n{label} Price Range (₹{min_p:.0f}K - ₹{max_p:.0f}K):")
        print(f"  - RMSE: ₹{range_rmse:.2f}K")
        print(f"  - MAE: ₹{range_mae:.2f}K")
        print(f"  - R²: {range_r2:.4f}")
        print(f"  - Sample size: {mask.sum()}")


### Analysis

The XGBoost model tells a different story about property pricing than does the Random Forest model, with significant ramifications for housing markets in the real world:

**1. Location is the Biggest Market Divider (60.6% of predictive power)**
- **West Mambalam and Premium Areas (36.7%)**:  A single location feature (West Mambalam and Premium areas) explains more than one-third of the predictions, implying that prices in some areas operate in a vacuum, separate from other features. This suggests property markets are not integrated markets, but rather micro-markets, determined by location.
- **Other Location Features (23.9%)**: The remaining location features (Veppampattu, Perungudi, T Nagar, etc.) combine to account for almost a quarter of the predictions, confirming that the main way markets are segmented is through location. Housing in prime locations are valued differently to housing in less desirable locations.

**2. Structural Features Become Secondary (4.8% of predictive power)**
- **Area Diminishment (3.6%)**: Contrary to Random Forest's revelation that area is the most important feature at 50.4%, XGBoost finds area at 3.6%. This implies that once location is known, area plays a much smaller role in determining price.
- **Bedrooms and Bathrooms (1.1% combined)**: These factors offer very little predictive power, suggesting they're trivial once location and floor area is determined. This reflects a real-world phenomenon where buyers in premium locations accept smaller unit sizes at premium prices, while buyers in standard areas prioritize space for their budget.

**3. Builder Reputation Emerges as Critical (34.6% of predictive power)**
- **Developer Dominance**: Builder importance has grown from 6.6% (Random Forest) to 34.6%, implying developers target different market segments with varying levels of pricing power. Viswaraj, JD Properties, and MEHTA REAL ESTATE have price premiums not captured by location and structure.
- **What This Means**: This probably reflects that developers operate in specific high-end markets, offer high-quality construction, or have strong brand loyalty. The model accounts for this through builder features because developers are proxies for unobservable quality and spatial concentration.

**4. Market Segmentation Hierarchy**
- The better cross-validation results of XGBoost (R2 = 0.8213 vs Random Forest's 0.7244) implies that house prices are governed by a hierarchy: Location defines market segment and price level; builder reputation defines premium/discount within the segment; and structural features only make fine adjustments to the price of the builder-location combination. This is in contrast to the additive model Linear Regression adopts.

**Key Takeaways for Stakeholders**

1. **For Buyers/Sellers**: Location choice is the most important decision (60% of price variation). After location, size provides little bargaining power. Builders' quality is crucial and can explain large price differences.

2. **For Investors**: Invest in micro-markets, not a market system. Investors should concentrate on pre-development location search: identifying new locations developers are focusing on. Developer choice is important once a location is found.

3. **For Developers**: Location quality trumps building quality. Being located in West Mambalam yields a 36.7% price advantage in the model, so developers should focus on buying land rather than on building cheaply.

## Final Comprehensive Analysis: Chennai Real Estate Market

### Summary

This analysis examines property pricing in the Chennai metropolitan region using machine learning models. The data shows that the real estate market is location fragmented with most of the property value being determined by geographic position whereas structural features are secondary factors. There is distinct stratification in the region between premium neighborhoods, standard areas, and developing suburbs.

---

### 1. Dataset Overview

**Dataset Composition:**
- Properties analyzed: 1,440+ (removal of outliers at 99th percentile)
- Price range: ₹20L to ₹600L+
- Key features: Area, BHK count, bathrooms, location (160+ neighborhoods)
- Geographic scope: Greater Chennai region

**Data Processing:**
- Log-transformed prices to reduce variance
- Removed top 1% outliers (luxury properties)
- Removed rows with missing values and unsuitable columns

---

### 2. Model Performance Comparison

| Metric | Linear Regression | Random Forest | XGBoost |
|--------|-------------------|---------------|---------|
| Test R² | 0.93 | 0.8401 | 0.8520 |
| Test RMSE | ₹22.94K | ₹34.40K | ₹33.10K |
| Test MAE | ₹10.73K | ₹11.81K | ₹11.42K |
| CV R² | 0.8002 | 0.7581 | 0.8213 |
| Overfitting Gap | 12.98 | 8.2 | 3.07 |

**Key Finding:** Linear Regression overfits severely. XGBoost achieves best generalization (3.07 gap) and highest CV R² (0.8213), while Random Forest balances performance with interpretability.

---

### 3. Feature Importance Analysis

#### Linear Regression: Top Impactful Locations

**Premium Locations (Positive Coefficients):**
- Tiruvottiyur: +0.805 (price premium)
- Gopalapuram: +0.789 (high-value neighborhood)

**Discounted Locations (Negative Coefficients):**
- Puzhal: -1.334 (strong discount)
- Sevvapet: -1.044 (moderate discount)
- Sriperumbudur: -0.938 (peripheral discount)

**Insight:** Clear price gradient from city center (premium) to periphery (discounted). The most significant linear predictor is location.

#### Random Forest: Feature Importance Distribution

- **Structural Features: 52.2%** (Area: 23.5%, BHK: 15.3%, Bathrooms: 13.4%)
- **Location Features: 31.9%** (Micro-market effects)
- **Builder Identity: 15.2%** (Developer reputation)

**Insight:** Area remains most important structural feature (23.5%), followed by location. Tree models are used to understand non-linear interactions between location and structure interacting multiplicatively.

#### XGBoost: Feature Importance Distribution

- **Location Features: 60.6%** (West Mambalam dominates at 36.7%)
- **Builder Identity: 34.7%** (Viswaraj: 15.5%, JD Properties: 6.4%, MEHTA: 3.8%)
- **Structural Features: 4.7%** (Area: 3.6%, BHK/Bathrooms: 1.1%)

**Key Insight:** XGBoost reveals a hierarchical market structure: location defines the market segment (60.6%), builder reputation commands premium within segment (34.7%), and area becomes secondary (3.6%).

---

### 4. Market Segmentation

The Chennai market has three different levels:

| Tier | Examples | Characteristics | Price Premium |
|------|----------|-----------------|----------------|
| **Premium** | West Mambalam, Veppampattu | Established infrastructure | +36% (XGBoost) |
| **Standard** | T Nagar, Perungudi | Mature neighborhoods | Baseline |
| **Developing** | Puzhal, Sriperumbudur | Emerging suburbs | -60-73% (Linear Regression) |

**Market Implication:** 160+ micro-markets operate independently. The analysis of XGBoost reveals that West Mambalam commands a price premium of around ~36% (0.367 feature importance). The linear regression location coefficients indicate much larger negative effects for some peripheral locations (for example: Puzhal ≈ -73.7%, Sevvapet ≈ -64.8%, Sriperumbudur ≈ -60.8% on the original price scale). Tree-based models (Random Forest and XGBoost) also show strong location dominance, but exact percent discounts vary by model and location and should be quantified by comparing location-level predicted prices when precise estimates are required.

---

### 5. Prediction Accuracy by Price Range

**Linear Regression Performance:**
- ₹0-₹100L: Tight clustering, high accuracy
- ₹100-₹200L: Moderate scatter, good predictions
- ₹200L+: Significant scatter, unreliable for luxury segment

**Random Forest Performance:**
- Low Range (₹15-43K): R² 0.5602, moderate accuracy
- Medium Range (₹43-72K): R² 0.2249, lower accuracy
- High Range (₹72-450K): R² 0.7385, good predictions for high-value properties

**XGBoost Performance:**
- Low Range (₹15-43K): R² 0.6578, solid accuracy
- Medium Range (₹43-72K): R² 0.1751, lower accuracy
- High Range (₹72-450K): R² 0.7581, excellent predictions for high-value properties

**Practical Note:** XGBoost shows no difference in performance between high and low price ranges, and performance is also better in the high-value segment (R² 0.7581 vs 0.7385 of random forest). Both tree-based models struggle in the medium price range (₹43-72K). XGBoost's better overall generalization makes it most reliable for diverse property portfolios.

---

### 6. Market Insights

**Infrastructure-Driven:** Central areas with established infrastructure command premiums; suburban areas with developing infrastructure offer appreciation potential.

**Spatial Fragmentation:** 160+ neighborhoods with different dynamics imply there is no mass appreciation; gains are concentrated in the developing regions.

**Market Maturity:** The existence of a clear price gradient and the formation of a micro-market is an indication that the market is becoming more mature and that there will be little growth in the premium areas but a high potential in the peripheral areas.

---

### 7. Model Reliability

#### Model Performance Summary

| Model | Test R² | CV R² | CV Gap | Best For |
|-------|---------|-------|--------|----------|
| **Linear Regression** | 0.93 | 0.8002 | 12.98 | Quick estimates (interpretable) |
| **Random Forest** | 0.8401 | 0.7581 | 8.2 | Balanced approach |
| **XGBoost** | 0.8520 | 0.8213 | 3.07 | Best performance & generalization |

#### When to Trust

- Standard property valuation (₹50-₹200L)
- Neighborhood price benchmarking
- Market trend analysis
- Portfolio valuation

#### When to Be Cautious

- Luxury properties (₹300L+)
- New developments
- Short-term forecasting
- High-stakes investment decisions

**Recommendation:** Use XGBoost for critical decisions (best generalization with 3.07 CV gap); Random Forest for balanced approach; Linear Regression only for quick estimates.

---

### Conclusion

The Chennai real estate market is essentially location-based and micro-segmented:

1. **Geography determines most of value** - Location is king
2. **Infrastructure is the value driver** - Access to infrastructure is valued highly
3. **160+ distinct micro-markets** - No homogenous market forces
4. **Size matters secondarily** - Important within location, not across locations
5. **Clear three-tier stratification** - Premium/Standard/Developing markets

#### Strategic Takeaways

- **Buyers:** Choose location first; expect price variation by neighborhood
- **Investors:** Early peripheral entry during infrastructure development offers best returns
- **Developers:** Secure premium locations; reputation provides up to 15% premiums

To succeed in Chennai's real estate, you need to know which of the market tiers you are in and to optimize within that micro-market environment.